# 3. Weighted KL modes and empirical covariance

Read [Chapter 5](../05-disorder.md). On a physical grid, the code diagonalizes
$\sqrt{W_x}C\sqrt{W_x}$, not the unweighted covariance. Shapes have units
$\mathrm{nm}^{-1/2}$, eigenvalues $\mathrm{meV}^2\,\mathrm{nm}$, and real
coefficients have variance equal to the eigenvalues. This notebook verifies
sample covariance against the **retained** covariance separately from KL
truncation. The seed is an educational verification seed.

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import json
import sys
import numpy as np
import os, tempfile
if "MPLCONFIGDIR" not in os.environ:
    _mpl_cache=tempfile.TemporaryDirectory(prefix="qwave-teaching-mpl-")
    os.environ["MPLCONFIGDIR"]=_mpl_cache.name
import matplotlib.pyplot as plt

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents)
            if (p / "pyproject.toml").is_file() and (p / "src/qwave").is_dir())
sys.path.insert(0, str(ROOT / "docs/derivations"))
from verification import VerificationBundle
plt.rcParams.update({"font.size": 10, "axes.grid": True, "grid.alpha": .2})

CONFIG = {'seeds': [730001], 'domain_nm': [-30.0, 30.0], 'grid_points': 121, 'sigma_mev': 2.0, 'correlation_length_nm': 5.0, 'kernel': 'gaussian', 'boundary_taper_nm': 0.0, 'retained_modes': 6, 'sample_count': 4096, 'approximations': ['spatial trapezoidal KL normalization', 'finite channel, zero disorder outside', 'reported roundoff clipping, no diagonal jitter', 'Gaussian PCG64 draws; no sample rescaling'], 'criteria': {'orthogonality_atol': 1e-12, 'covariance_relative_atol': 1e-12, 'empirical_error_expected_rms_limit': 4.0, 'omitted_variance_standard_errors': 5.0}}

output = Path(globals().get("RUN_OUTPUT", ROOT / "results" /
              ("learning-03-" + datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ"))))
bundle = VerificationBundle(output, CONFIG, scope='Gaussian covariance and KL numerical verification; no transport performance inference')
print(json.dumps(CONFIG, indent=2))


In [ ]:
from qwave.disorder import covariance_matrix,covariance_modes,sample_disorder,covariance_statistics,lag_correlations
x=np.linspace(*CONFIG["domain_nm"],CONFIG["grid_points"])
C=covariance_matrix(x,**{key:CONFIG[key] for key in ["sigma_mev","correlation_length_nm","kernel","boundary_taper_nm"]})
modes=covariance_modes(x,C,retained_modes=CONFIG["retained_modes"])
samples=sample_disorder(modes,sample_count=CONFIG["sample_count"],seed=CONFIG["seeds"][0])
stats=covariance_statistics(samples.retained_mev,modes.retained_covariance)
full_stats=covariance_statistics(samples.full_mev,C)
phi=modes.shapes_per_sqrt_nm;lam=modes.eigenvalues_mev2_nm;weights=modes.weights_nm
orth_error=float(np.max(abs(phi.T@(weights[:,None]*phi)-np.eye(len(x)))))
reconstruction_error=float(np.linalg.norm((phi*lam)@phi.T-C)/np.linalg.norm(C))
omitted_per_draw=((samples.full_mev-samples.retained_mev)**2)@weights
expected_omission=float(lam[CONFIG["retained_modes"]:].sum())
omission_se=float(np.std(omitted_per_draw,ddof=1)/np.sqrt(len(omitted_per_draw)))
omission_error=float(abs(np.mean(omitted_per_draw)-expected_omission))
bundle.json("kl-diagnostics",modes.diagnostics)
bundle.json("retained-covariance-statistics",stats["summary"])
bundle.json("full-covariance-statistics",full_stats["summary"])
bundle.json("omitted-variance",{"expected_mev2_nm":expected_omission,"measured_mev2_nm":float(np.mean(omitted_per_draw)),
 "sample_standard_error_mev2_nm":omission_se,"absolute_error_mev2_nm":omission_error})
bundle.check("weighted orthonormality",orth_error<1e-12,measured=orth_error,criterion="Max |Phi.T W Phi - I| < 1e-12")
bundle.check("full covariance reconstruction",reconstruction_error<1e-12,measured=reconstruction_error,
 criterion="Relative Frobenius error <1e-12, with clipped negative roundoff mass reported")
bundle.check("empirical retained covariance",stats["summary"]["frobenius_error_in_expected_sampling_rms"]<4.,
 measured=stats["summary"],criterion="Error <4 expected Gaussian sampling RMS; a broad regression guard, not a simultaneous confidence interval")
bundle.check("mean omitted integrated variance",omission_error<5*omission_se,
 measured={"error_mev2_nm":omission_error,"standard_error_mev2_nm":omission_se},
 criterion="Mean omitted weighted norm agrees with sum of omitted eigenvalues within 5 estimated standard errors")
bundle.arrays("covariance",units={"x_nm":"nm","weights_nm":"nm","C":"meV^2","C_retained":"meV^2",
 "empirical_C":"meV^2","empirical_full_C":"meV^2","eigenvalues":"meV^2 nm","phi":"nm^-1/2",
 "z":"1","eta":"meV sqrt(nm)","full_samples":"meV","retained_samples":"meV"},
 x_nm=x,weights_nm=weights,C=C,C_retained=modes.retained_covariance,empirical_C=stats["empirical_covariance_mev2"],
 empirical_full_C=full_stats["empirical_covariance_mev2"],eigenvalues=lam,phi=phi,z=samples.standard_normals,
 eta=samples.coefficients_mev_sqrt_nm,full_samples=samples.full_mev,retained_samples=samples.retained_mev)
bundle.table("eigenvalues",[{"mode":i+1,"lambda_mev2_nm":value,"cumulative_fraction":float(lam[:i+1].sum()/lam.sum())}
                             for i,value in enumerate(lam)])
print(json.dumps({"retained_fraction":modes.diagnostics["retained_variance_fraction"],"empirical":stats["summary"]},indent=2))


In [ ]:
fig,axs=plt.subplots(2,2,figsize=(10,7),layout="constrained")
positive=lam>0
axs[0,0].semilogy(np.arange(1,len(lam)+1)[positive],lam[positive],"o-",ms=3)
axs[0,0].axvline(CONFIG["retained_modes"],ls="--",label="Retained K")
axs[0,0].set(xlabel="Mode index (1)",ylabel="Covariance eigenvalue (meV^2 nm)");axs[0,0].legend()
for k in range(3):axs[0,1].plot(x,modes.retained_shapes[:,k],label=f"Mode {k+1}")
axs[0,1].set(xlabel="Position (nm)",ylabel="Normalized mode shape (1/sqrt(nm))");axs[0,1].legend()
for draw in range(3):axs[1,0].plot(x,samples.full_mev[draw],label=f"Draw {draw+1}")
axs[1,0].set(xlabel="Position (nm)",ylabel="Disorder potential (meV)");axs[1,0].legend()
lag=np.arange(len(x))*(x[1]-x[0])
axs[1,1].plot(lag,lag_correlations(C),label="Prescribed full covariance")
axs[1,1].plot(lag,lag_correlations(full_stats["empirical_covariance_mev2"]),label="Empirical full draws")
axs[1,1].plot(lag,lag_correlations(modes.retained_covariance),"--",label="Retained covariance")
axs[1,1].set(xlabel="Separation (nm)",ylabel="Mean normalized correlation (1)");axs[1,1].legend(fontsize=8)
bundle.figure(fig,"covariance-modes","Actual covariance spectrum, weighted mode shapes, three stored Gaussian draws and correlation comparison. Positive eigenvalues only on logarithmic axis; all eigenvalues including clipped zeros are saved. Truncation changes covariance and is not corrected by rescaling.")
fig,axs=plt.subplots(1,3,figsize=(11,3.5),layout="constrained")
for ax,matrix,title in zip(axs,[C,modes.retained_covariance,stats["empirical_covariance_mev2"]],
                           ["Prescribed full C","Retained C_K","Empirical retained C"]):
    im=ax.imshow(matrix,origin="lower",extent=[x[0],x[-1],x[0],x[-1]],vmin=-.5,vmax=4.,cmap="coolwarm")
    ax.set(xlabel="Position x (nm)",ylabel="Position x' (nm)",title=title)
fig.colorbar(im,ax=axs,label="Covariance (meV^2)")
bundle.figure(fig,"covariance-samples","Distinct full, retained, and empirical retained covariance matrices. The last comparison tests finite sampling; the first comparison tests KL representation error.")
bundle.finish()
